In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import os

from dataloader_brand import get_brand_dataloaders
from model_brand_resnet import get_resnet18_brand_model

In [ ]:
# 하이퍼파라미터 설정
batch_size = 32
num_epochs = 10
learning_rate = 1e-4
save_path = 'best_brand_model.pth'

# 장치 설정 (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 데이터 불러오기
train_loader, val_loader, _ = get_brand_dataloaders(
    data_dir="C:/Users/Admin/Desktop/food_brand_dataset",  # ← 사용자에 맞게 경로 수정
    batch_size=batch_size,
    num_workers=2
)

# 모델 불러오기
model = get_resnet18_brand_model(num_classes=9, pretrained=True).to(device)

# 손실 함수 & 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 학습 루프
best_val_acc = 0.0
for epoch in range(num_epochs):
    print(f"\n[Epoch {epoch+1}/{num_epochs}]")

    # 학습 단계
    model.train()
    train_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    avg_train_loss = train_loss / total
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f}")

    # 검증 단계
    model.eval()
    val_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    avg_val_loss = val_loss / total
    print(f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

    #  가장 성능 좋은 모델 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"📦 Best model saved to: {save_path}")